## Columns/Properties

Each feature in the GeoJSON has these properties, accessible by properties.{name}. These correspond to a column in the CSV as well. 

| name | data type | description | notes | to do |
|------|-----------|-------------|-------|-------|
| OBJECTID | OID | Internal feature number.; Description source; ESRI; Description of values Sequential unique whole numbers that are automatically generated. |  | drop |
| SIFCODE1 | String | Street Code for first intersection retrieved from (SIF); Description source; SIF | order is arbitrary |  |
| SIFCODE2 | String | Street Code for Second intersection retrieved from (SIF); Description source; SIF | "  |  |
| INTID | String | ID for the intersection (SCCAD_ID+SIFCODE1+SIFCODE2) | unique id for intersection | use as index |
| SCCAD_ID | Integer | Unique ID number used in Hansen | not unique | drop |
| FST_INTPRE | String | Street Direction (N, S, E, W) for first intersection | FST prefix | Combine with other FST_... columns for compact roadname |
| FST_INTNAME | String | Street Name for first intersection |  | " |
| FST_INTSUF | String | Street Type for first intersection | FST suffix. Ln, Rd, St, Blvd. etc | " |
| SEC_INTPRE | String | Street Direction (N, S, E, W) for second intersection | SEC prefix | Combine with other SEC_... columns for compact roadname  |
| SEC_INTNAME | String | Street Name for second intersection |  | " |
| SEC_INTSUF | String | Street Type for second intersection | SEC suffix. Similar to FST suffix. | " |
| FST_SIFID | Integer |  | Integer representation for SIFCODE1 |  |
| SEC_SIFID | Integer |  | Integet representation for SIFCODE2 |  |



## Geometry 

Data / objects are more complicated:

These coordinates work with a KY State official projection: `NAD_1983_StatePlane_Kentucky_North_FIPS_1601_Feet`
`LOJIC projection: ESRI:102679`

source: https://www.lojic.org/sites/default/files/metadata/address_intersection.htm



| name | data type | description | notes | to do |
|------|-----------|-------------|-------|-------|
| X_COORD | Double | X coordinates for the intersection | State plane coordinates. |  |
| X *(CSV only)* | Double | X coordinates for the intersection | State plane coordinates. |  |
| Y_COORD | Double | Y coordinates for the intersection | State plane coordinates. |  |
| Y *(CSV only)* | Double | Y coordinates for the intersection | State plane coordinates. |  |


GeoJSON stores some geometry information separately from other properties that were listed above.

The GeoJSON geometry object is a (longitude, latitude) point using a standard projection: `epsg:4326`

| name | data type | description | notes | to do |
|------|-----------|-------------|-------|-------|
| geometry.type | String | Type of geometry object. | Always "Point" | drop |
| geometry.geometry | tuple | Feature geometry.; Description source; ESRI; Description of values Coordinates defining the features. | Point where roads intersect: (Longitude, Latitude)  |  |

## Extent 
Geographic extent / Bounding rectangle 

Extent type  

* Extent used for searching
```py
west_longitude = -85.945347
east_longitude = -85.344499
north_latitude = 38.378034
south_latitude = 38.005894
```

* Extent in the item's coordinate system 
```py
west_longitude_co = 1154395.500000
east_longitude_co = 1325086.990000
south_latitude_co = 188677.437500
north_latitude_co = 321629.781250
```


In [ ]:
# scraping metadata from LOJIC

# imports
import re
import pandas as pd
from os import path

# constants
metadata_path = "~/code/county_coverage/data/raw/intersections/intersections.info"

'/Users/bencampbell/code/county_coverage/data/raw/intersections/intersections.info'

In [83]:
# parsing centerline metadata 
FIELDNAME = re.compile('\s*Field\s*([A-Z0-9_]+)')
FIELDID = re.compile("Field\s*([A-Z0-9_]+).*")

def parse_fieldname_data(metadata_lines):
    page = list()
    name = 'never appears' 
        
    # break up file to sections 
    for line in metadata_lines:
        line = line.strip()
        m = FIELDID.match(line)
        if m:
            name = m.groups()[0]
            info = list()
            memo = {"name": name, 'info':info}
            page.append(memo)
        elif line.startswith("Hide"):
            assert name in line
        elif line: # ignore empty lines
            info.append(line) 

    # parse infomation in each section
    for section_data in page:
        info = section_data.pop('info')
        index = 0
        for line in info:
            index += 1
            if line.startswith("*\u2009"):
                key, value = line.lstrip("*\u2009").split("\u2003", maxsplit=1)
                section_data[key.lower()] = value
            elif line.startswith("Field description"):
                break
        section_data['description'] = info[index:]
    return page

def get_fieldname_metadata(filepath):
    with open(filepath) as file:
        out = parse_fieldname_data(file)
    return pd.DataFrame.from_dict(out)

intersection_metadata_raw = get_fieldname_metadata(path.expanduser(metadata_path))
#all(fieldname_metadata['name'] == fieldname_metadata['alias']) # good news


In [84]:
intersection_metadata_raw.head()

,name,alias,data type,width,precision,scale,description
0,OBJECTID,OBJECTID,OID,4,10,0,"[Internal feature number., Description source,..."
1,SHAPE,SHAPE,Geometry,4,0,0,"[Feature geometry., Description source, ESRI, ..."
2,SIFCODE1,SIFCODE1,String,4,0,0,[Street Code for first intersection retrieved ...
3,SIFCODE2,SIFCODE2,String,4,0,0,[Street Code for Second intersection retrieved...
4,INTID,INTID,String,14,0,0,[ID for the intersection (SCCAD_ID+SIFCODE1+SI...


In [78]:

relevant_columns = ['name', 'data type', 'description']
intersection_metadata = intersection_metadata_raw[['name', 'data type', 'description']].copy()
intersection_metadata.description = intersection_metadata.description.apply("; ".join)
for name in ['notes', 'to do']:
    intersection_metadata[name] = ''
intersection_metadata.head()

,name,data type,description,notes,to do
0,OBJECTID,OID,Internal feature number.; Description source; ...,,
1,SHAPE,Geometry,Feature geometry.; Description source; ESRI; D...,,
2,SIFCODE1,String,Street Code for first intersection retrieved f...,,
3,SIFCODE2,String,Street Code for Second intersection retrieved ...,,
4,INTID,String,ID for the intersection (SCCAD_ID+SIFCODE1+SIF...,,


In [2]:
import os
os.getcwd()

'/Users/bencampbell/code/county_coverage'

In [79]:
def create_markdown(data):
    labels, _, *lines = data.to_markdown().splitlines()  
    labels = " | ".join(label.strip() for label in labels.split('|')).strip()
    sep = ''.join(('-' if char != "|" else "|") for char in labels)
    out = [labels, sep]
    for line in lines:
        nl = " | ".join(text.strip() for text in line.split("|"))
        out.append(nl.strip())
    return out

def write_markdown(data):
    print(*create_markdown(data), sep='\n')
#print(intersection_meta_filtered.to_markdown())
write_markdown(intersection_metadata)

|  | name | data type | description | notes | to do |
|--|------|-----------|-------------|-------|-------|
| 0 | OBJECTID | OID | Internal feature number.; Description source; ESRI; Description of values Sequential unique whole numbers that are automatically generated. |  |  |
| 1 | SHAPE | Geometry | Feature geometry.; Description source; ESRI; Description of values Coordinates defining the features. |  |  |
| 2 | SIFCODE1 | String | Street Code for first intersection retrieved from (SIF); Description source; SIF |  |  |
| 3 | SIFCODE2 | String | Street Code for Second intersection retrieved from (SIF); Description source; SIF |  |  |
| 4 | INTID | String | ID for the intersection (SCCAD_ID+SIFCODE1+SIFCODE2) |  |  |
| 5 | SCCAD_ID | Integer | Unique ID number used in Hansen |  |  |
| 6 | FST_INTPRE | String | Street Direction (N, S, E, W) for first intersection |  |  |
| 7 | FST_INTNAME | String | Street Name for first intersection |  |  |
| 8 | FST_INTSUF | String | Street Type for

In [ ]:


#Extent 
#Geographic extent 
#Bounding rectangle 
#Extent type  Extent used for searching
west_longitude = -85.945347
east_longitude = -85.344499
north_latitude = 38.378034
south_latitude = 38.005894
#* Extent contains the resource Yes

#Extent in the item's coordinate system 
west_longitude_co = 1154395.500000
east_longitude_co = 1325086.990000
south_latitude_co = 188677.437500
north_latitude_co = 321629.781250
#* Extent contains the resource Yes